Data inladen (nieuwe verdeling van test en train)

In [19]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Bepaal hoe de foto in het model moet (64x64 en zwart-wit)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), # Zwart-wit voor lichter model
    transforms.Resize((64, 64)),                 # Formaat terugbrengen naar 64x64
    transforms.ToTensor()                        # Omzetten naar PyTorch Tensors
])

# 2. Datasets inladen via relatieve paden (werkt op elke pc binnen de projectmap)

# --- AXIAAL ---
train_data_ax = datasets.ImageFolder(root="Resplit_80_20/Training/Axiaal", transform=transform)
test_data_ax = datasets.ImageFolder(root="Resplit_80_20/Testing/Axiaal", transform=transform)

# --- SAGITTAAL ---
train_data_sag = datasets.ImageFolder(root="Resplit_80_20/Training/Sagittaal", transform=transform)
test_data_sag = datasets.ImageFolder(root="Resplit_80_20/Testing/Sagittaal", transform=transform)

# --- CORONAAL ---
train_data_cor = datasets.ImageFolder(root="Resplit_80_20/Training/Coronaal", transform=transform)
test_data_cor = datasets.ImageFolder(root="Resplit_80_20/Testing/Coronaal", transform=transform)

# 3. DataLoaders aanmaken (stuurt de data in stapeltjes naar het model tijdens trainen)
batch_size = 32

train_loader_ax = DataLoader(train_data_ax, batch_size=batch_size, shuffle=True)
test_loader_ax = DataLoader(test_data_ax, batch_size=batch_size, shuffle=False)

train_loader_sag = DataLoader(train_data_sag, batch_size=batch_size, shuffle=True)
test_loader_sag = DataLoader(test_data_sag, batch_size=batch_size, shuffle=False)

train_loader_cor = DataLoader(train_data_cor, batch_size=batch_size, shuffle=True)
test_loader_cor = DataLoader(test_data_cor, batch_size=batch_size, shuffle=False)

print("Alle data is succesvol ingeladen per aanzicht!")
print(f"Gevonden klassen (labels): {train_data_ax.classes}")

Alle data is succesvol ingeladen per aanzicht!
Gevonden klassen (labels): ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


Model 1- CNN

In [24]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import classification_report # Toegevoegd voor de matrix

# 1. Bouw het lichtgewicht CNN
class SimpelCNN(nn.Module):
    def __init__(self, aantal_klassen):
        super(SimpelCNN, self).__init__()
        # Kijkt in 2D naar de zwart-wit foto
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Maakt de uiteindelijke beslissing (4096 -> 4 tumor klassen)
        self.fc1 = nn.Linear(16 * 32 * 32, aantal_klassen)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.view(x.size(0), -1) # Interne platslag vóór de eindbeslissing
        x = self.fc1(x)
        return x

# 2. Model instellen
aantal_klassen = len(train_data_ax.classes) # Automatisch 4 klassen
cnn_model = SimpelCNN(aantal_klassen)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# 3. Het model trainen (5 rondes)
epochs = 10
print(f"CNN aan het trainen voor {epochs} rondes...")

for epoch in range(epochs):
    cnn_model.train()
    running_loss = 0.0
    for images, labels in train_loader_ax:
        optimizer.zero_grad()           # Reset de berekeningen
        outputs = cnn_model(images)     # Laat het model raden
        loss = criterion(outputs, labels) # Bereken de fout
        loss.backward()                 # Leer van de fout
        optimizer.step()                # Pas het model aan
        running_loss += loss.item()
        
    print(f"Ronde (Epoch) {epoch+1}/{epochs} afgerond. Loss: {running_loss/len(train_loader_ax):.4f}")

# 4. Testen en resultaat printen
cnn_model.eval()
correct = 0
totaal = 0

# Nieuwe lijsten om de data voor de matrix op te slaan
voorspellingen_ax = []
echte_labels_ax = []

with torch.no_grad(): # Tijdens testen hoeft hij niet meer te leren
    for images, labels in test_loader_ax:
        outputs = cnn_model(images)
        _, voorspeld = torch.max(outputs.data, 1)
        
        totaal += labels.size(0)
        correct += (voorspeld == labels).sum().item()
        
        # Sla op voor de scikit-learn matrix
        voorspellingen_ax.extend(voorspeld.numpy())
        echte_labels_ax.extend(labels.numpy())

# Haal het rapport op als data (dictionary) in plaats van standaard tekst
rapport = classification_report(echte_labels_ax, voorspellingen_ax, target_names=train_data_ax.classes, output_dict=True)

print("--- Evaluatiematrix Custom CNN (Axiaal) ---")
print(f"{'Tumorsoort':<18} precision   recall   f1-score")
print("-" * 48)

# Forceer het printen van alle klassen in jouw mappenstructuur
for klasse in train_data_ax.classes:
    if klasse in rapport:
        p = rapport[klasse]['precision']
        r = rapport[klasse]['recall']
        f = rapport[klasse]['f1-score']
        print(f"{klasse:<18} {p:<11.2f} {r:<8.2f} {f:.2f}")

print("-" * 48)
print(f"{'accuracy':<18} {' ':<11} {' ':<8} {rapport['accuracy']:.2f}")

CNN aan het trainen voor 10 rondes...
Ronde (Epoch) 1/10 afgerond. Loss: 1.2729
Ronde (Epoch) 2/10 afgerond. Loss: 0.9104
Ronde (Epoch) 3/10 afgerond. Loss: 0.7817
Ronde (Epoch) 4/10 afgerond. Loss: 0.6338
Ronde (Epoch) 5/10 afgerond. Loss: 0.5475
Ronde (Epoch) 6/10 afgerond. Loss: 0.4693
Ronde (Epoch) 7/10 afgerond. Loss: 0.4105
Ronde (Epoch) 8/10 afgerond. Loss: 0.3637
Ronde (Epoch) 9/10 afgerond. Loss: 0.3211
Ronde (Epoch) 10/10 afgerond. Loss: 0.2889
--- Evaluatiematrix Custom CNN (Axiaal) ---
Tumorsoort         precision   recall   f1-score
------------------------------------------------
glioma_tumor       0.87        0.90     0.88
meningioma_tumor   0.90        0.89     0.90
no_tumor           0.92        0.89     0.90
pituitary_tumor    0.97        0.98     0.97
------------------------------------------------
accuracy                                0.90


In [25]:
# Haal het rapport op als data (dictionary) in plaats van standaard tekst
rapport = classification_report(echte_labels_ax, voorspellingen_ax, target_names=train_data_ax.classes, output_dict=True)

print("--- Evaluatiematrix Custom CNN (Axiaal) ---")
print(f"{'Tumorsoort':<18} precision   recall   f1-score")
print("-" * 48)

# Forceer het printen van alle klassen in jouw mappenstructuur
for klasse in train_data_ax.classes:
    if klasse in rapport:
        p = rapport[klasse]['precision']
        r = rapport[klasse]['recall']
        f = rapport[klasse]['f1-score']
        print(f"{klasse:<18} {p:<11.2f} {r:<8.2f} {f:.2f}")

print("-" * 48)
print(f"{'accuracy':<18} {' ':<11} {' ':<8} {rapport['accuracy']:.2f}")

# Voeg hier de macro average toe
macro_p = rapport['macro avg']['precision']
macro_r = rapport['macro avg']['recall']
macro_f = rapport['macro avg']['f1-score']
print(f"{'macro avg':<18} {macro_p:<11.2f} {macro_r:<8.2f} {macro_f:.2f}")

--- Evaluatiematrix Custom CNN (Axiaal) ---
Tumorsoort         precision   recall   f1-score
------------------------------------------------
glioma_tumor       0.87        0.90     0.88
meningioma_tumor   0.90        0.89     0.90
no_tumor           0.92        0.89     0.90
pituitary_tumor    0.97        0.98     0.97
------------------------------------------------
accuracy                                0.90
macro avg          0.91        0.91     0.91


cnn-sagittaal

In [21]:
# 1. Model instellen voor Sagittaal
aantal_klassen = len(train_data_sag.classes)
cnn_model_sag = SimpelCNN(aantal_klassen) 
optimizer_sag = optim.Adam(cnn_model_sag.parameters(), lr=0.001)

# 2. Het model trainen
epochs = 10
print(f"Sagittaal CNN aan het trainen voor {epochs} rondes...")

for epoch in range(epochs):
    cnn_model_sag.train()
    running_loss = 0.0
    for images, labels in train_loader_sag:
        optimizer_sag.zero_grad()
        outputs = cnn_model_sag(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_sag.step()
        running_loss += loss.item()
        
    print(f"Ronde {epoch+1}/{epochs} afgerond. Loss: {running_loss/len(train_loader_sag):.4f}")

# 3. Testen, accuraatheid en evaluatiematrix genereren
cnn_model_sag.eval()
correct = 0
totaal = 0
voorspellingen_sag = []
echte_labels_sag = []

with torch.no_grad():
    for images, labels in test_loader_sag:
        outputs = cnn_model_sag(images)
        _, voorspeld = torch.max(outputs.data, 1)
        
        totaal += labels.size(0)
        correct += (voorspeld == labels).sum().item()
        
        voorspellingen_sag.extend(voorspeld.numpy())
        echte_labels_sag.extend(labels.numpy())

# Haal het rapport op als data (dictionary) in plaats van standaard tekst
rapport_sag = classification_report(echte_labels_sag, voorspellingen_sag, target_names=train_data_sag.classes, output_dict=True)

print("--- Schone Evaluatiematrix Custom CNN (Sagittaal) ---")
print(f"{'Tumorsoort':<18} precision   recall   f1-score")
print("-" * 48)

# Forceer het printen van alle klassen in jouw mappenstructuur
for klasse in train_data_sag.classes:
    if klasse in rapport_sag:
        p = rapport_sag[klasse]['precision']
        r = rapport_sag[klasse]['recall']
        f = rapport_sag[klasse]['f1-score']
        print(f"{klasse:<18} {p:<11.2f} {r:<8.2f} {f:.2f}")

print("-" * 48)
print(f"{'accuracy':<18} {' ':<11} {' ':<8} {rapport_sag['accuracy']:.2f}")


Sagittaal CNN aan het trainen voor 10 rondes...
Ronde 1/10 afgerond. Loss: 1.1278
Ronde 2/10 afgerond. Loss: 0.7372
Ronde 3/10 afgerond. Loss: 0.6000
Ronde 4/10 afgerond. Loss: 0.4933
Ronde 5/10 afgerond. Loss: 0.4563
Ronde 6/10 afgerond. Loss: 0.3773
Ronde 7/10 afgerond. Loss: 0.3238
Ronde 8/10 afgerond. Loss: 0.2810
Ronde 9/10 afgerond. Loss: 0.2675
Ronde 10/10 afgerond. Loss: 0.2331
--- Schone Evaluatiematrix Custom CNN (Sagittaal) ---
Tumorsoort         precision   recall   f1-score
------------------------------------------------
glioma_tumor       0.98        0.86     0.91
meningioma_tumor   0.84        0.98     0.90
no_tumor           1.00        0.43     0.60
pituitary_tumor    0.98        0.98     0.98
------------------------------------------------
accuracy                                0.93


cnn- coronaal

In [22]:
# 1. Model instellen voor Coronaal
aantal_klassen = len(train_data_cor.classes)
cnn_model_cor = SimpelCNN(aantal_klassen) 
optimizer_cor = optim.Adam(cnn_model_cor.parameters(), lr=0.001)

# 2. Het model trainen
epochs = 10
print(f"Coronaal CNN aan het trainen voor {epochs} rondes...")

for epoch in range(epochs):
    cnn_model_cor.train()
    running_loss = 0.0
    for images, labels in train_loader_cor:
        optimizer_cor.zero_grad()
        outputs = cnn_model_cor(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_cor.step()
        running_loss += loss.item()
        
    print(f"Ronde {epoch+1}/{epochs} afgerond. Loss: {running_loss/len(train_loader_cor):.4f}")

# 3. Testen, accuraatheid en evaluatiematrix genereren
cnn_model_cor.eval()
correct = 0
totaal = 0
voorspellingen_cor = []
echte_labels_cor = []

with torch.no_grad():
    for images, labels in test_loader_cor:
        outputs = cnn_model_cor(images)
        _, voorspeld = torch.max(outputs.data, 1)
        
        totaal += labels.size(0)
        correct += (voorspeld == labels).sum().item()
        
        voorspellingen_cor.extend(voorspeld.numpy())
        echte_labels_cor.extend(labels.numpy())

# Haal het rapport op als data (dictionary) in plaats van standaard tekst
rapport_cor = classification_report(echte_labels_cor, voorspellingen_cor, target_names=train_data_cor.classes, output_dict=True)

print("--- Schone Evaluatiematrix Custom CNN (Coronaal) ---")
print(f"{'Tumorsoort':<18} precision   recall   f1-score")
print("-" * 48)

# Forceer het printen van alle klassen in jouw mappenstructuur
for klasse in train_data_cor.classes:
    if klasse in rapport_cor:
        p = rapport_cor[klasse]['precision']
        r = rapport_cor[klasse]['recall']
        f = rapport_cor[klasse]['f1-score']
        print(f"{klasse:<18} {p:<11.2f} {r:<8.2f} {f:.2f}")

print("-" * 48)
print(f"{'accuracy':<18} {' ':<11} {' ':<8} {rapport_cor['accuracy']:.2f}")


Coronaal CNN aan het trainen voor 10 rondes...
Ronde 1/10 afgerond. Loss: 1.0303
Ronde 2/10 afgerond. Loss: 0.6413
Ronde 3/10 afgerond. Loss: 0.4955
Ronde 4/10 afgerond. Loss: 0.3809
Ronde 5/10 afgerond. Loss: 0.2960
Ronde 6/10 afgerond. Loss: 0.2681
Ronde 7/10 afgerond. Loss: 0.2095
Ronde 8/10 afgerond. Loss: 0.1734
Ronde 9/10 afgerond. Loss: 0.1777
Ronde 10/10 afgerond. Loss: 0.1387
--- Schone Evaluatiematrix Custom CNN (Coronaal) ---
Tumorsoort         precision   recall   f1-score
------------------------------------------------
glioma_tumor       0.96        0.93     0.95
meningioma_tumor   0.91        0.97     0.94
no_tumor           1.00        0.71     0.83
pituitary_tumor    1.00        1.00     1.00
------------------------------------------------
accuracy                                0.95


MODEL 2- random forrest

In [23]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Hulpfunctie om PyTorch data plat te slaan voor scikit-learn (Random Forest)
def prepareer_voor_rf(dataloader):
    X_lijst, y_lijst = [], []
    for images, labels in dataloader:
        # Sla de 64x64 tensor plat naar een rij van 4096 getallen
        images_plat = images.view(images.size(0), -1).numpy()
        X_lijst.append(images_plat)
        y_lijst.append(labels.numpy())
    
    # Plak alle kleine stapeltjes aan elkaar tot één grote dataset
    return np.vstack(X_lijst), np.concatenate(y_lijst)

# 2. Haal de Axiale data op en sla het plat
print("Data voorbereiden voor Random Forest...")
X_train_rf, y_train_rf = prepareer_voor_rf(train_loader_ax)
X_test_rf, y_test_rf = prepareer_voor_rf(test_loader_ax)

# 3. Train het Random Forest Model
print("Random Forest aan het trainen (dit is zo klaar)...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_rf, y_train_rf)

# 4. Testen en resultaat printen
rf_voorspellingen = rf_model.predict(X_test_rf)
rf_accuracy = accuracy_score(y_test_rf, rf_voorspellingen)

print(f"\n--- Resultaat Random Forest (Axiaal) ---")
print(f"Nauwkeurigheid (Accuracy): {rf_accuracy * 100:.2f}%")
print(classification_report(y_test_rf, rf_voorspellingen, target_names=train_data_ax.classes))

Data voorbereiden voor Random Forest...
Random Forest aan het trainen (dit is zo klaar)...

--- Resultaat Random Forest (Axiaal) ---
Nauwkeurigheid (Accuracy): 97.10%
                  precision    recall  f1-score   support

    glioma_tumor       1.00      0.96      0.98       115
meningioma_tumor       0.96      0.96      0.96       139
        no_tumor       0.94      0.98      0.96       135
 pituitary_tumor       1.00      1.00      1.00        59

        accuracy                           0.97       448
       macro avg       0.98      0.97      0.98       448
    weighted avg       0.97      0.97      0.97       448

